In [ ]:
import os
import glob
import pandas as pd

# Define input (Silver) and output (Bronze) directory paths
silver_dir = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Silver\AllRecords"
bronze_dir = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze"

# Create Bronze directory if it doesn't already exist
if not os.path.exists(bronze_dir):
    os.makedirs(bronze_dir)
    print(f"Created directory: {bronze_dir}")

# Find all Excel and CSV files inside the Silver folder
file_patterns = [
    os.path.join(silver_dir, "*.xlsx"),
    os.path.join(silver_dir, "*.xls"),
    os.path.join(silver_dir, "*.csv")
]

all_files = []
for pattern in file_patterns:
    all_files.extend(glob.glob(pattern))

print(f"Found {len(all_files)} file(s) in Silver layer:")
for f in all_files:
    print(f" - {os.path.basename(f)}")

if not all_files:
    print("\n❌ No matching Excel or CSV files were found in the Silver folder.")
else:
    dataframes = []

    # Read each file and append to our list
    for file_path in all_files:
        try:
            filename = os.path.basename(file_path)
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)
            
            # --- Drop existing S. No. columns if present ---
            sno_variations = ['S. No.', 'S.No.', 'S.No', 'S. No', 'SNo', 'S_No', 's.no.', 's.no']
            cols_to_drop = [col for col in df.columns if str(col).strip() in sno_variations]
            if cols_to_drop:
                df.drop(columns=cols_to_drop, inplace=True)

            # Optionally add a metadata column tracking the source filename
            df['Source_File'] = filename
            
            dataframes.append(df)
            print(f"✓ Successfully loaded: {filename} ({len(df)} rows)")
        except Exception as e:
            print(f"❌ Error reading {os.path.basename(file_path)}: {e}")

    # Merge/Concatenate all DataFrames into one
    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)
        
        # --- Generate new continuous S. No. column ---
        merged_df.insert(0, 'S. No.', range(1, len(merged_df) + 1))

        # Define output destination file path
        output_file_path = os.path.join(bronze_dir, "RAW_MERGED.xlsx")
        
        # Save merged dataframe to Excel
        merged_df.to_excel(output_file_path, index=False)
        print(f"\n✅ Merge complete! Total combined rows: {len(merged_df)}")
        print(f"📁 Output file created at:\n   {output_file_path}")
    else:
        print("\n❌ Failed to parse any data from the identified files.")

Found 19 file(s) in Silver layer:
 - MOEFCC.xlsx
 - SEIAA_ANDAMAN.xlsx
 - SEIAA_ANDHRA.xlsx
 - SEIAA_ARUNACHAL.xlsx
 - SEIAA_ASSAM.xlsx
 - SEIAA_BENGAL.xlsx
 - SEIAA_BIHAR.xlsx
 - SEIAA_CHHATTISGARH.xlsx
 - SEIAA_GUJARAT.xlsx
 - SEIAA_JHARKHAND.xlsx
 - SEIAA_KA.xlsx
 - SEIAA_KERALA.xlsx
 - SEIAA_MAHARASHTRA.xlsx
 - SEIAA_MP.xlsx
 - SEIAA_Multiple.xlsx
 - SEIAA_ODISHA.xlsx
 - SEIAA_TELANGANA.xlsx
 - SEIAA_TN.xlsx
 - SEIAA_UP.xlsx
✓ Successfully loaded: MOEFCC.xlsx (8875 rows)
✓ Successfully loaded: SEIAA_ANDAMAN.xlsx (13 rows)
✓ Successfully loaded: SEIAA_ANDHRA.xlsx (3716 rows)
✓ Successfully loaded: SEIAA_ARUNACHAL.xlsx (17 rows)
✓ Successfully loaded: SEIAA_ASSAM.xlsx (1573 rows)
✓ Successfully loaded: SEIAA_BENGAL.xlsx (1418 rows)
✓ Successfully loaded: SEIAA_BIHAR.xlsx (2346 rows)
✓ Successfully loaded: SEIAA_CHHATTISGARH.xlsx (2878 rows)
✓ Successfully loaded: SEIAA_GUJARAT.xlsx (9156 rows)
✓ Successfully loaded: SEIAA_JHARKHAND.xlsx (1842 rows)
✓ Successfully loaded: SEIAA_KA.xls